# Teil B: Datenintegration, Bereinigung und Geodaten

**Ziel:** Die Echtzeitdaten des Parkleitsystems (PLS) enthalten nur die *freien* Plätze. Um die Auslastung zu berechnen, braucht es die **Kapazität** jedes Parkhauses. Dieses Notebook kombiniert dafür mehrere Quellen, dokumentiert den Abgleich, bereinigt die Rohdaten und stellt die Auslastung auf einer Karte dar.

| Quelle | Zugriff | Inhalt |
|---|---|---|
| PLS-Feed der Stadt Zürich | Web-API (RSS/XML), alle 15 Min. gesammelt | freie Plätze, Status |
| Stadt Zürich: „Öffentlich zugängliche Parkhäuser“ | Web-API (WFS, GeoJSON) | Kapazität, Koordinaten |
| Detailseiten auf pls-zh.ch | Web Scraping (BeautifulSoup) | Kapazität für fehlende Parkhäuser |
| swisstopo-Adresssuche | Web-API | Koordinaten für fehlende Parkhäuser |

**Inhalt**
1. Stadt-Datensatz laden
2. Abgleich mit dem Feed über den Namen
3. Manuelle Zuordnung und Web Scraping → `garages.csv`
4. Bereinigung der Rohdaten → `occupancy_clean.csv`
5. Karte der durchschnittlichen Auslastung

**Ergebnis:** `data/processed/garages.csv` (Stammdaten) und `data/processed/occupancy_clean.csv` (bereinigte Auslastung).

## Schritt 1: Stadt-Datensatz laden (Web-API)

Der Datensatz „Öffentlich zugängliche Parkhäuser“ der Stadt Zürich wird über die WFS-Schnittstelle als GeoJSON abgerufen. Mit `SRSNAME=EPSG:4326` erhalten wir GPS-Koordinaten (Breiten- und Längengrad). Pro Parkhaus übernehmen wir Name, Adresse, Kapazität (`anzahl_oeffentliche_pp`), Typ und Koordinaten.

**Ergebnis:** 136 Parkhäuser, gespeichert als Rohdaten in `data/external/city_garages.csv`.

In [1]:
import os
import requests
import pandas as pd

# Web-API (WFS) der Stadt Zürich: alle öffentlich zugänglichen Parkhäuser als GeoJSON
url = ("https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Oeffentlich_zugaengliche_Parkhaeuser"
       "?SERVICE=WFS&VERSION=1.1.0&REQUEST=GetFeature&TYPENAME=poi_parkhaus_view"
       "&OUTPUTFORMAT=application/vnd.geo%2Bjson&SRSNAME=EPSG:4326")
geo = requests.get(url, timeout=60).json()

# Jedes "feature" ist ein Parkhaus: Eigenschaften + Punkt-Koordinaten
rows = []
for feature in geo["features"]:
    p = feature["properties"]
    lon, lat = feature["geometry"]["coordinates"]
    rows.append({
        "city_name": p["name"].strip(),
        "city_address": p["adresse"],
        "capacity": p["anzahl_oeffentliche_pp"],
        "category": p["kategorie"],
        "latitude": lat,
        "longitude": lon,
    })

city = pd.DataFrame(rows)

# Rohdaten speichern (eigener Ordner, damit der Roboter nicht reinkommt)
os.makedirs("../data/external", exist_ok=True)
city.to_csv("../data/external/city_garages.csv", index=False)

print(city.shape)
city.head(10)

(136, 6)


,city_name,city_address,capacity,category,latitude,longitude
0,A-Park,Albisriederstrasse 348,74.0,Parkhaus,47.375637,8.486451
1,Accu,Fritz-Heeb-Weg 5,213.0,Parkhaus,47.414853,8.540591
2,Airgate,Leutschenbachstrasse 37,163.0,Parkhaus,47.415035,8.552979
3,Albisriederdörfli,Bachwiesenstrasse 141,26.0,Parkhaus,47.376036,8.485043
4,Albisriederplatz,Badenerstrasse 380c,65.0,Parkhaus,47.379393,8.508120
5,Aldi,Ueberlandstrasse 441,58.0,Parkhaus,47.405736,8.592106
6,Aldi Seebach,Schärenmoosstrasse 77a,70.0,Parkhaus,47.419748,8.556535
7,Altstetten-Luggweg,Luggwegstrasse 107a,44.0,Parkhaus,47.385053,8.490689
8,Atrium 565,Badenerstrasse 565b,18.0,Parkhaus,47.383777,8.497597
9,Bellevue-Garage,Stadelhoferstrasse 42,20.0,Parkhaus,47.367742,8.546466


## Schritt 2: Abgleich mit dem Feed über den Namen

Die beiden Quellen haben **keinen gemeinsamen Schlüssel**: Der Feed verwendet eine `parking_id` (z. B. `cp`), die Stadt nur Namen. Zudem sind die Namen unterschiedlich geschrieben (z. B. „Parkhaus Max-Bill-Platz“ vs. „Max Bill-Platz“).

**Vorgehen:**
- **2.1** Alle Parkhäuser aus den Feed-Dateien laden, jede `parking_id` einmal.
- **2.2** Namen in beiden Quellen normalisieren: Präfix „Parkhaus“/„Parkplatz“ entfernen, Kleinschreibung, Umlaute ersetzen, Sonderzeichen entfernen.
- **2.3** Beide Tabellen über den normalisierten Namen mit einem **Left Join** verbinden. So bleiben alle 37 Feed-Parkhäuser erhalten, auch ohne Treffer.

**Ergebnis:** 29 von 37 Parkhäusern automatisch zugeordnet, 8 ohne Treffer.

*Hinweis:* Bei mehreren Treffern stimmen die Adressen nicht überein (z. B. Accu: Otto-Schütz-Weg vs. Fritz-Heeb-Weg 5). Das liegt vermutlich an unterschiedlichen Ein- und Ausfahrten; der Name ist deshalb der zuverlässigere Schlüssel.

In [2]:
import glob
import re

# 1) Alle Parkhäuser aus dem Feed: jede parking_id einmal (aus allen Monats-CSVs)
feed_files = sorted(glob.glob("../data/raw/parking_*.csv"))
feed_raw = pd.concat([pd.read_csv(f) for f in feed_files], ignore_index=True)
feed = (feed_raw[["parking_id", "parking_name", "address"]]
        .drop_duplicates(subset="parking_id", keep="last")
        .reset_index(drop=True))
print("Feed-Parkhäuser:", len(feed))

# 2) Namen vereinheitlichen, damit unterschiedliche Schreibweisen zusammenpassen
def normalize(name):
    name = re.sub(r"^(Parkhaus|Parkplatz)\s+", "", str(name).strip())
    name = name.lower()
    for a, b in [("ä", "ae"), ("ö", "oe"), ("ü", "ue"), ("é", "e")]:
        name = name.replace(a, b)
    return re.sub(r"[^a-z0-9]", "", name)

feed["key"] = feed["parking_name"].apply(normalize)
city["key"] = city["city_name"].apply(normalize)

# 3) Abgleich: alle Feed-Parkhäuser behalten (left join)
matched = feed.merge(city, on="key", how="left")

found = matched["capacity"].notna().sum()
print(f"Automatisch zugeordnet: {found} von {len(feed)}")
print("\nNicht zugeordnet:")
matched.loc[matched["capacity"].isna(), ["parking_id", "parking_name", "address"]]

Feed-Parkhäuser: 37
Automatisch zugeordnet: 29 von 37

Nicht zugeordnet:


,parking_id,parking_name,address
10,hafen_enge,Parkhaus Hafen Enge,Mythenquai 21
18,messe,Parkhaus Messe Zürich AG,Andreasstrasse 65
28,unispital_nord,Parkhaus USZ Nord,Frauenklinikstrasse
29,uni_irchel,Parkhaus Uni Irchel,Winterthurerstrasse 181
32,zueri11,Parkhaus Züri 11 Shopping,Nansenstrasse 5/7
34,theater_11,Parkplatz Theater 11,Dörfli-/Thurgauerstrasse
35,unispital_sued,Parkplatz USZ Süd,Gloriastrasse
36,puls5,Puls 5 Parkgarage,Giessereistrasse 18


## Schritt 3: Manuelle Zuordnung und Web Scraping

Die 8 fehlenden Parkhäuser werden ergänzt:

- **3.1 Manuell (5):** Gleiches Parkhaus, aber stark abweichender Name, z. B. „Parkhaus Uni Irchel“ = „Universität Irchel“.
- **3.2 Web Scraping (2):** Theater 11 und USZ Süd fehlen im Stadt-Datensatz. Ihre Kapazität wird mit `requests` und `BeautifulSoup` von der Detailseite auf pls-zh.ch gelesen („Angebot: … Plätze“). Zwischen den Anfragen wird 1 Sekunde gewartet, um den Server nicht zu belasten.
- **3.3** Übersicht, wie jedes Parkhaus zugeordnet wurde (Spalte `match_method`).
- **3.4** Garagen-Tabelle speichern.

| match_method | Anzahl | Bedeutung |
|---|---|---|
| name | 29 | automatisch über den normalisierten Namen |
| manual | 5 | von Hand zugeordnet |
| scraped | 2 | Kapazität per Web Scraping |
| none | 1 | Hafen Enge: keine Detailseite, liefert seit Wochen keine Werte |

**Ergebnis:** 36 von 37 Parkhäusern haben eine Kapazität → `data/processed/garages.csv`.

In [5]:
import time

# 4) Manuelle Zuordnung: gleiche Parkhäuser, aber andere Namen im Stadt-Datensatz
manual = {
    "messe": "Messe",
    "puls5": "Puls 5",
    "uni_irchel": "Universität Irchel",
    "unispital_nord": "Unispital Nord",
    "zueri11": "Züri 11",
}
city_by_name = city.set_index("city_name", drop=False)
matched["match_method"] = matched["capacity"].notna().map({True: "name", False: None})

for pid, city_name in manual.items():
    i = matched.index[matched["parking_id"] == pid][0]
    for col in ["city_name", "city_address", "capacity", "category", "latitude", "longitude"]:
        matched.loc[i, col] = city_by_name.loc[city_name, col]
    matched.loc[i, "match_method"] = "manual"

# 5) Web Scraping: Kapazität von der Detailseite auf pls-zh.ch für den Rest
from bs4 import BeautifulSoup

def scrape_capacity(pid):
    url = f"https://www.pls-zh.ch/parkhaus/{pid}.jsp?pid={pid}"
    r = requests.get(url, headers={"User-Agent": "ZHAW-student-project"}, timeout=30)
    text = BeautifulSoup(r.text, "html.parser").get_text(" ", strip=True)  # HTML -> reiner Text
    m = re.search(r"Angebot:\s*(\d+)\s*Pl", text)
    return int(m.group(1)) if m else None

for i in matched.index[matched["capacity"].isna()]:
    pid = matched.loc[i, "parking_id"]
    cap = scrape_capacity(pid)
    time.sleep(1)  # höflich: nicht zu schnell hintereinander anfragen
    if cap:
        matched.loc[i, "capacity"] = cap
        matched.loc[i, "match_method"] = "scraped"
    else:
        matched.loc[i, "match_method"] = "none"

# 6) Übersicht für die Dokumentation
print(matched["match_method"].value_counts())

# 7) Garagen-Tabelle speichern (für A und C)
garages = matched[["parking_id", "parking_name", "capacity", "latitude", "longitude",
                   "city_name", "city_address", "match_method"]]
os.makedirs("../data/processed", exist_ok=True)
garages.to_csv("../data/processed/garages.csv", index=False)
garages

match_method
name       29
manual      5
scraped     2
none        1
Name: count, dtype: int64


,parking_id,parking_name,capacity,latitude,longitude,city_name,city_address,match_method
0,accu,Parkhaus Accu,213.0,47.414853,8.540591,Accu,Fritz-Heeb-Weg 5,name
1,albisriederplatz,Parkhaus Albisriederplatz,65.0,47.379393,8.508120,Albisriederplatz,Badenerstrasse 380c,name
2,bleicherweg,Parkhaus Bleicherweg,258.0,47.367372,8.535696,Bleicherweg,Beethovenstrasse 35,name
3,center_11,Parkhaus Center Eleven,377.0,47.412826,8.540275,Center Eleven,Sophie-Taeuber-Strasse 4,name
4,cp,Parkhaus City Parking,607.0,47.375690,8.535496,City Parking,Gessnerallee 14,name
5,cityport,Parkhaus Cityport,131.0,47.410875,8.540895,Cityport,Affolternstrasse 56,name
6,crowne_plaza,Parkhaus Crowne Plaza,310.0,47.381153,8.506867,Crowne Plaza,Bienenstrasse 9,name
7,dorflinde,Parkhaus Dorflinde,48.0,47.407595,8.548751,Dorflinde,Schwamendingenstrasse 31,name
8,feldegg,Parkhaus Feldegg,147.0,47.360383,8.553527,Feldegg,Riesbachstrasse 6,name
9,globus,Parkhaus Globus,139.0,47.376055,8.537717,Globus,Löwenstrasse 50a,name


## Schritt 4: Bereinigung der Rohdaten

Die Regeln wurden aus einer Analyse der gesammelten Daten abgeleitet:

| Regel | Problem in den Daten | Entscheidung | Entfernte Zeilen |
|---|---|---|---|
| 1 | Status `unknown`, keine freien Plätze | Zeile entfernen | 2'686 |
| 2 | P West und Züri 11 melden seit August denselben Wert | Werte entfernen, die seit **über 48 h** unverändert sind | 2'293 |
| 3 | Messe: Anzeige endet bei 999 (in 83 % der Zeilen); Hafen Enge: keine Kapazität | beide Parkhäuser ausschliessen | 2'381 |
| 4 | Doppelte Snapshots (Collector zweimal gestartet) | pro 15-Minuten-Fenster und Parkhaus nur den ersten behalten | 2'904 |
| 5 | Bei 7 Parkhäusern meldet der Feed mehr freie Plätze als die Stadt-Kapazität | Kapazität = Maximum aus Stadt-Angabe und höchstem gemeldetem Wert | 0 (angepasst) |

**Warum 48 h und nicht 30 Minuten?** Die Quellzeit im Feed ist der Zeitpunkt der *letzten Änderung*, nicht der letzten Messung. Nachts ändert sich wenig: Um 3 Uhr sind 73 % der Werte „älter als 30 Minuten“, aber trotzdem korrekt. Eine kurze Grenze hätte fast alle Nachtwerte gelöscht.

Danach wird die Auslastung berechnet:

$$\text{Auslastung} = \frac{\text{Kapazität} - \text{freie Plätze}}{\text{Kapazität}}$$

Die Zeitmerkmale (Stunde, Wochentag, Wochenende) werden in **Lokalzeit Zürich** berechnet, inklusive Umstellung auf Winterzeit am 25.10.

**Ergebnis:** Von 68'006 Rohzeilen bleiben 57'742 saubere Zeilen für 33 Parkhäuser → `data/processed/occupancy_clean.csv` (Stand: 9. November 2026).

In [8]:
# ===== Schritt 5: Bereinigung =====
raw = pd.concat([pd.read_csv(f) for f in sorted(glob.glob("../data/raw/parking_*.csv"))], ignore_index=True)
garages = pd.read_csv("../data/processed/garages.csv").drop(columns=["max_free", "capacity_used"], errors="ignore")
raw["fetched_at_utc"] = pd.to_datetime(raw["fetched_at_utc"], utc=True)
raw["source_updated_at_utc"] = pd.to_datetime(raw["source_updated_at_utc"], utc=True)

log = [("Rohdaten", len(raw))]  # Protokoll: Zeilen nach jeder Regel
df = raw.copy()

# Regel 1: Zeilen ohne Messwert
df = df[(df["status"] != "unknown") & df["free_spaces"].notna()]
log.append(("1: ohne Messwert entfernt", len(df)))

# Regel 2: Eingefrorene Werte (seit über 48 h keine Änderung gemeldet)
df["lag_min"] = (df["fetched_at_utc"] - df["source_updated_at_utc"]).dt.total_seconds() / 60
df = df[df["lag_min"] <= 48 * 60]
log.append(("2: eingefrorene Werte (> 48 h) entfernt", len(df)))

# Regel 3: Hafen Enge (keine Kapazität) und Messe (Anzeige endet bei 999)
df = df[~df["parking_id"].isin(["hafen_enge", "messe"])]
log.append(("3: Hafen Enge und Messe entfernt", len(df)))

# Regel 4: Duplikate im selben 15-Minuten-Fenster
df["slot_utc"] = df["fetched_at_utc"].dt.floor("15min")
df = df.sort_values("fetched_at_utc").drop_duplicates(subset=["slot_utc", "parking_id"], keep="first")
log.append(("4: Duplikate pro 15-Min.-Fenster entfernt", len(df)))

# Regel 5: Kapazität nie kleiner als die höchste gemeldete Zahl freier Plätze
max_free = df.groupby("parking_id")["free_spaces"].max().rename("max_free")
garages = garages.merge(max_free, on="parking_id", how="left")
garages["capacity_used"] = garages[["capacity", "max_free"]].max(axis=1)
adjusted = garages[garages["max_free"] > garages["capacity"]]
print("Kapazität angepasst bei:", ", ".join(adjusted["parking_id"]))

# Auslastung und Zeitmerkmale (Lokalzeit Zürich)
df = df.merge(garages[["parking_id", "capacity_used"]], on="parking_id", how="left")
df["occupancy_rate"] = (df["capacity_used"] - df["free_spaces"]) / df["capacity_used"]
local = df["slot_utc"].dt.tz_convert("Europe/Zurich")
df["date"] = local.dt.date
df["hour"] = local.dt.hour
df["weekday"] = local.dt.day_name()
df["is_weekend"] = local.dt.day_of_week >= 5

# Protokoll anzeigen
log = pd.DataFrame(log, columns=["Schritt", "Zeilen"])
log["entfernt"] = -log["Zeilen"].diff().fillna(0).astype(int)
print(log.to_string(index=False))

# Speichern
clean = df[["slot_utc", "parking_id", "status", "free_spaces", "capacity_used", "occupancy_rate",
            "date", "hour", "weekday", "is_weekend"]]
clean.to_csv("../data/processed/occupancy_clean.csv", index=False)
garages.to_csv("../data/processed/garages.csv", index=False)
print("\nSaubere Daten:", clean.shape, "| Parkhäuser:", clean["parking_id"].nunique())

Kapazität angepasst bei: cityport, helvetia, jungholz, octavo, stampfenbach, urania, puls5
                                  Schritt  Zeilen  entfernt
                                 Rohdaten   68006         0
                1: ohne Messwert entfernt   65320      2686
  2: eingefrorene Werte (> 48 h) entfernt   63027      2293
         3: Hafen Enge und Messe entfernt   60646      2381
4: Duplikate pro 15-Min.-Fenster entfernt   57742      2904

Saubere Daten: (57742, 10) | Parkhäuser: 33


## Schritt 5: Karte der durchschnittlichen Auslastung

Für Theater 11 und USZ Süd fehlen Koordinaten. Sie werden über die Adresssuche von swisstopo (Web-API) ermittelt. Bei USZ Süd ist die Position nur **ungefähr**, weil der Feed keine Hausnummer nennt.

Die Karte (folium, Kartengrundlage © OpenStreetMap-Mitwirkende) zeigt jedes Parkhaus als Kreis:
- **Farbe:** durchschnittliche Auslastung (grün = meist leer, rot = meist voll)
- **Grösse:** Kapazität

**Erste Beobachtung:** Die am stärksten ausgelasteten Parkhäuser liegen in der Innenstadt und am Seeufer (Zürichhorn, Crowne Plaza, Utoquai, Talgarten, je über 70 %), während Parkhäuser in Oerlikon und den Aussenquartieren deutlich weniger belegt sind (z. B. Max-Bill-Platz, Jungholz, Cityport unter 20 %).

Die interaktive Karte ist gespeichert unter `figures/map_avg_occupancy.html`.

In [10]:
# ===== Schritt 6: Karte =====
import folium
import branca.colormap as cm

garages = pd.read_csv("../data/processed/garages.csv")
clean = pd.read_csv("../data/processed/occupancy_clean.csv")

# Fehlende Koordinaten (Theater 11, USZ Süd) über die Adresssuche von swisstopo holen
def geocode(address):
    url = "https://api3.geo.admin.ch/rest/services/api/SearchServer"
    params = {"searchText": address, "type": "locations", "sr": 4326, "limit": 1}
    attrs = requests.get(url, params=params, timeout=30).json()["results"][0]["attrs"]
    return attrs["lat"], attrs["lon"]

missing_addresses = {
    "theater_11": "Thurgauerstrasse 7, 8050 Zürich",
    "unispital_sued": "Gloriastrasse 5, 8006 Zürich",   # ungefähr: der Feed nennt keine Hausnummer
}
for pid, address in missing_addresses.items():
    lat, lon = geocode(address)
    garages.loc[garages["parking_id"] == pid, ["latitude", "longitude"]] = [lat, lon]
    garages.loc[garages["parking_id"] == pid, "match_method"] = "scraped + geocoded"
garages.to_csv("../data/processed/garages.csv", index=False)

# Durchschnittliche Auslastung pro Parkhaus
avg = clean.groupby("parking_id")["occupancy_rate"].mean().rename("avg_occupancy")
map_df = garages.merge(avg, on="parking_id", how="inner").dropna(subset=["latitude", "longitude"])
print("Parkhäuser auf der Karte:", len(map_df))

# Karte: Farbe = Auslastung (grün -> gelb -> rot), Grösse = Kapazität
colormap = cm.LinearColormap(["#2E7D32", "#F2A900", "#C62828"], vmin=0, vmax=1,
                             caption="Durchschnittliche Auslastung")
zurich = folium.Map(location=[47.385, 8.535], zoom_start=13, tiles="OpenStreetMap")
zurich.fit_bounds(map_df[["latitude", "longitude"]].values.tolist())   # auf alle Parkhäuser zoomen
for _, g in map_df.iterrows():
    folium.CircleMarker(
        location=[g["latitude"], g["longitude"]],
        radius=4 + g["capacity_used"] ** 0.5 / 2,
        color=colormap(g["avg_occupancy"]), fill=True, fill_opacity=0.8, weight=1,
        tooltip=f"{g['parking_name']}: {g['avg_occupancy']:.0%} belegt, {int(g['capacity_used'])} Plätze",
    ).add_to(zurich)
colormap.add_to(zurich)

os.makedirs("../figures", exist_ok=True)
zurich.save("../figures/map_avg_occupancy.html")
zurich

Parkhäuser auf der Karte: 33


## Limitationen

- **Kapazität:** Die Quellen widersprechen sich (z. B. Urania: Stadt 480 Plätze, pls-zh.ch 607, davon 479 Kurzparking). Die Auslastung hängt von dieser Wahl ab.
- **Niedrige Auslastung** bei einzelnen Parkhäusern (Max-Bill-Platz ⌀ 12 %) kann real sein oder auf eine zu hohe Kapazität hinweisen.
- **Lücken:** Der Collector fiel mehrmals aus (21 Lücken über 30 Minuten, längste ca. 6 h). Diese Lücken werden bewusst nicht aufgefüllt.
- **Koordinaten** von USZ Süd sind nur ungefähr.
- **Stichtag:** Die bereinigten Daten enthalten die Messungen bis zum letzten Ausführen des Notebooks. Für die finale Analyse wird das Notebook am vereinbarten Stichtag neu ausgeführt.